In [ ]:

import re
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


# 1. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

torch.manual_seed(42)
random.seed(42)


# 2. Parameters
CSV_PATH = "IMDB Dataset.csv"

MAX_VOCAB_SIZE = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 128
HIDDEN_DIM = 64
BATCH_SIZE = 64
EPOCHS = 5


# 3. Load CSV file using Pandas
print("\nLoading IMDB dataset...")

df = pd.read_csv(CSV_PATH)

print(df.head())
print("Dataset shape:", df.shape)


# 4. Select text and label columns
# Common IMDB CSV columns: review and sentiment
# Alternative columns: text and label

if "review" in df.columns:
    text_column = "review"
elif "text" in df.columns:
    text_column = "text"
else:
    raise ValueError("Review/text column not found in CSV")

if "sentiment" in df.columns:
    label_column = "sentiment"
elif "label" in df.columns:
    label_column = "label"
else:
    raise ValueError("Sentiment/label column not found in CSV")


# Remove missing values
df = df.dropna(subset=[text_column, label_column]).copy()

# Convert sentiment into binary labels
def convert_label(value):

    if str(value).strip().lower() in ["positive", "pos", "1"]:
        return 1

    elif str(value).strip().lower() in ["negative", "neg", "0"]:
        return 0

    else:
        raise ValueError("Unexpected sentiment label: " + str(value))


df["label"] = df[label_column].apply(convert_label)

texts = df[text_column].astype(str).tolist()
labels = df["label"].tolist()

print("\nTotal reviews:", len(texts))
print("Positive reviews:", labels.count(1))
print("Negative reviews:", labels.count(0))


# 5. Split dataset into training, validation and testing
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print("\nTraining samples:", len(train_texts))
print("Validation samples:", len(val_texts))
print("Testing samples:", len(test_texts))


# 6. Tokenization and vocabulary creation
def tokenize(text):

    return re.findall(r"\b\w+\b", text.lower())


counter = Counter()

for review in train_texts:
    counter.update(tokenize(review))


vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in counter.most_common(MAX_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))


# 7. Create IMDB Dataset class
class IMDBDataset(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        tokens = tokenize(self.texts[index])[:MAX_LENGTH]

        encoded = [
            vocab.get(word, 1)
            for word in tokens
        ]

        if len(encoded) == 0:
            encoded = [1]

        text = torch.tensor(encoded, dtype=torch.long)

        label = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return text, label


# 8. Create datasets
train_dataset = IMDBDataset(train_texts, train_labels)
val_dataset = IMDBDataset(val_texts, val_labels)
test_dataset = IMDBDataset(test_texts, test_labels)


# 9. Padding function
def collate_batch(batch):

    texts, labels = zip(*batch)

    lengths = torch.tensor([
        len(text) for text in texts
    ])

    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=0
    )

    labels = torch.stack(labels)

    return texts, lengths, labels


# 10. Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)


# 11. Build LSTM model
class LSTMClassifier(nn.Module):

    def __init__(self):

        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(
            len(vocab),
            EMBEDDING_DIM,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True
        )

        self.fc = nn.Linear(HIDDEN_DIM, 1)

    def forward(self, text, lengths):

        embedded = self.embedding(text)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        output, (hidden, cell) = self.lstm(packed)

        hidden = hidden[-1]

        output = self.fc(hidden)

        return output.squeeze(1)


model = LSTMClassifier().to(device)

print("\nLSTM Model:")
print(model)


# 12. Loss function and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# 13. Train LSTM model
train_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for texts, lengths, labels in train_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts, lengths)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = (outputs >= 0).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct / total

    train_losses.append(avg_loss)
    train_accuracies.append(train_accuracy)


    # Validation
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for texts, lengths, labels in val_loader:

            texts = texts.to(device)
            labels = labels.to(device)

            outputs = model(texts, lengths)

            predictions = (outputs >= 0).float()

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = 100 * val_correct / val_total

    val_accuracies.append(val_accuracy)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Accuracy: {train_accuracy:.2f}% "
        f"Validation Accuracy: {val_accuracy:.2f}%"
    )


# 14. Test model evaluation
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for texts, lengths, labels in test_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts, lengths)

        predictions = (outputs >= 0).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total

print("\nFinal Test Accuracy:", round(test_accuracy, 2), "%")


# 15. Plot accuracy and loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(train_accuracies, label="Training Accuracy", marker="o")
plt.plot(val_accuracies, label="Validation Accuracy", marker="o")

plt.title("LSTM Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)

plt.plot(train_losses, label="Training Loss", marker="o")

plt.title("LSTM Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


# 16. Example predictions
def predict_sentiment(review):

    model.eval()

    tokens = tokenize(review)[:MAX_LENGTH]

    encoded = [
        vocab.get(word, 1)
        for word in tokens
    ]

    if len(encoded) == 0:
        encoded = [1]

    text = torch.tensor(
        encoded,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    lengths = torch.tensor([len(encoded)])

    with torch.no_grad():

        output = model(text, lengths)

        probability = torch.sigmoid(output).item()

    sentiment = "Positive" if probability >= 0.5 else "Negative"

    return sentiment, probability


sample_indices = random.sample(
    range(len(test_texts)),
    min(5, len(test_texts))
)

probabilities = []

for i, index in enumerate(sample_indices):

    review = test_texts[index]
    actual_label = test_labels[index]

    sentiment, probability = predict_sentiment(review)

    probabilities.append(probability)

    actual = "Positive" if actual_label == 1 else "Negative"

    print(f"\nSample {i+1}")
    print("Review:", review[:150], "...")
    print("Actual Sentiment:", actual)
    print("Predicted Sentiment:", sentiment)
    print("Confidence:", round(probability, 4))


# 17. Visualize example predictions
plt.figure(figsize=(9, 5))

colors = [
    "green" if p >= 0.5 else "red"
    for p in probabilities
]

plt.bar(
    range(1, len(probabilities) + 1),
    probabilities,
    color=colors
)

plt.axhline(
    y=0.5,
    color="black",
    linestyle="--",
    label="Decision Threshold"
)

plt.xlabel("Review Number")
plt.ylabel("Positive Sentiment Probability")
plt.title("LSTM Example Predictions")

plt.xticks(range(1, len(probabilities) + 1))
plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y")

plt.show()

print("\nPractical Completed Successfully!")
